In [ ]:
from google.colab import drive

#mounting Google Drive to save outputs permanently
drive.mount('/content/drive')

#creating a folder in the Drive specifically for this thesis project
import os

#setting the HuggingFace token so datasets can be downloaded
# HF_TOKEN not required: all models used here are public

os.makedirs('/content/drive/MyDrive/rag-thesis/models/saved_models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/rag-thesis/results', exist_ok=True)
print("Google Drive mounted and thesis folders created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and thesis folders created.


In [2]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'ECS7001P_Assignment3_exp4_multimodal_QA_PernilleBergesen (2).ipynb', '.ipynb_checkpoints', 'rag-thesis']


In [3]:
!git clone https://github.com/pernillejorg/retrieval-effects-claim-verification.git
%cd retrieval-effects-claim-verification
!git checkout step2.1-baseline-model

fatal: destination path 'rag-claim-verification' already exists and is not an empty directory.
/content/retrieval-effects-claim-verification
Already on 'step2.1-baseline-model'
Your branch is up to date with 'origin/step2.1-baseline-model'.


In [4]:
!pip install -r requirements.txt
!pip install "datasets==2.21.0" -q

In [ ]:
import os, shutil

#where load_scifact_open expects the files (inside the cloned repo)
target = '/content/retrieval-effects-claim-verification/data/scifact_open/cache'
os.makedirs(target, exist_ok=True)

#where it was uploaded in Drive
source = '/content/drive/MyDrive/rag-thesis/data/scifact_open/cache'

#copying every file that was uploaded into the repo's cache folder
for fname in os.listdir(source):
    shutil.copy(os.path.join(source, fname), os.path.join(target, fname))
    print(f"copied {fname}")

print("SciFact-Open cache in place.")

copied corpus_candidates.jsonl
copied claims_metadata.jsonl
copied claims.jsonl
copied corpus.jsonl
SciFact-Open cache in place.


In [ ]:
#1. training the SciFact baseline (trains + saves checkpoint)
!python models/baseline.py --dataset scifact 2>&1 | tee results/baseline_scifact_log.txt

#2. evaluating that trained baseline on SciFact-Open (zero-shot, no training)
!python models/baseline.py --dataset scifact_open 2>&1 | tee results/baseline_scifact_open_log.txt


  RoBERTa No-Retrieval Baseline  --  SCIFACT

Using device: cuda

The repository for allenai/scifact contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/allenai/scifact.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
Generating train split: 100%|██████████| 5183/5183 [00:00<00:00, 16894.19 examples/s]
Train claims : 1261
Val claims   : 450
Train label distribution: {'NEI': 304, 'CONTRADICT': 341, 'SUPPORT': 616}

Loading tokenizer: roberta-base

Checking claim token lengths...
  Maximum claim token length in this split: 75
  MAX_LENGTH=128 is safe -- no claims will be truncated.

--- Learning rate search ---

  Trying learning rate: 1e-05
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4747.66it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Sta

In [ ]:
#saving to drive 
import shutil, os

#1. saving the SciFact model checkpoint to Drive (only SciFact has a trained model)
shutil.copytree(
    '/content/retrieval-effects-claim-verification/models/saved_models/baseline_scifact',
    '/content/drive/MyDrive/rag-thesis/models/saved_models/baseline_scifact',
    dirs_exist_ok=True
)
print("SciFact model saved to Drive.")

#2. saving ALL results (SciFact values + SciFact-Open values, JSON and txt logs) to Drive
os.makedirs('/content/drive/MyDrive/rag-thesis/results', exist_ok=True)
shutil.copytree(
    '/content/retrieval-effects-claim-verification/results',
    '/content/drive/MyDrive/rag-thesis/results',
    dirs_exist_ok=True
)
print("All results (SciFact + SciFact-Open) saved to Drive.")

SciFact model saved to Drive.
All results (SciFact + SciFact-Open) saved to Drive.
